# Multi-agent blog pipeline with LangChain + LangGraph

This notebook is a **small, readable** multi-agent system:

- **Supervisor agent** — the **only** LangGraph node; it **invokes** the specialists below when state still needs their work.
- **ResearchAgent** — a sub-agent (not a graph node): **only** gathers web context with **Tavily**.
- **WritingAgent** — a sub-agent (not a graph node): drafts the blog and saves it under **`temp/`**.

**Concepts you will see:** shared state (`TypedDict`), **sub-agents** as Python objects with `.invoke(state)`, a **supervisor agent** as the **only** LangGraph node that calls those specialists, and a **linear graph** (`START → supervisor → END`).

### Flow (supervisor invokes sub-agents)

The **StateGraph** has one worker node: the **supervisor agent**. Research and writing are **not** separate graph nodes — they are **`ResearchAgent`** and **`WritingAgent`** classes whose `.invoke(state)` the supervisor calls.

```
  START ──► ┌─────────────────────────────────────┐ ──► END
            │  supervisor_agent (LangGraph node)   │
            │    ├─► research.invoke(state)  …    │
            │    └─► writer.invoke(state)    …    │
            └─────────────────────────────────────┘
```

### Setup

In your project `.env` (repo root), set:

- `TAVILY_API_KEY` — used by the research tool.
- `OPENAI_API_KEY` — used by the writing agent (any `ChatOpenAI`-compatible provider key works with the same env name).

Install dependencies (once):

```bash
pip install langgraph langchain langchain-openai langchain-community python-dotenv tavily-python
```

Run the notebook with the working directory at the **Agents-101** project root so `temp/` is created next to `requirements.txt`.

In [9]:
from __future__ import annotations

import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

load_dotenv()


def project_root() -> Path:
    """Resolve repo root whether the kernel cwd is the repo or agents_demos/."""
    cwd = Path.cwd().resolve()
    if (cwd / "agents_demos").is_dir():
        return cwd
    if cwd.name == "agents_demos" and (cwd.parent / "requirements.txt").is_file():
        return cwd.parent
    return cwd


ROOT = project_root()
TEMP_BLOG_DIR = ROOT / "temp"
TEMP_BLOG_DIR.mkdir(parents=True, exist_ok=True)
print("Blog output directory:", TEMP_BLOG_DIR)

Blog output directory: C:\Users\mistr\Documents\PP\Agents-101\temp


### 1. Shared state — the contract between agents

LangGraph passes a **single state object** through every node. Each node returns a **partial update** (a dict); LangGraph merges updates into the state.

We keep fields explicit so it is obvious what each agent reads and writes:

| Field | Who writes it | Purpose |
|-------|----------------|---------|
| `topic` | You (input) | Blog subject |
| `research_notes` | Research agent | Raw findings from Tavily |
| `blog_markdown` | Writing agent | Final article text |
| `blog_file_path` | Writing agent | Where the file was saved |
| `supervisor_trace` | Supervisor agent | Log of which sub-agent ran |

In [10]:
class BlogState(TypedDict):
    topic: str
    research_notes: str
    blog_markdown: str
    blog_file_path: str
    supervisor_trace: list[str]


def initial_state(topic: str) -> BlogState:
    return BlogState(
        topic=topic,
        research_notes="",
        blog_markdown="",
        blog_file_path="",
        supervisor_trace=[],
    )


def safe_slug(text: str, max_len: int = 48) -> str:
    s = re.sub(r"[^\w\s-]", "", text.lower())
    s = re.sub(r"[-\s]+", "-", s).strip("-")
    return (s[:max_len] or "post").rstrip("-")

### 2. Sub-agents — specialists, not LangGraph nodes

`ResearchAgent` and `WritingAgent` are small **agent objects**: each exposes `.invoke(state) -> dict` and owns one job (Tavily vs. drafting + file). They are **not** registered with `add_node`.

### 3. Supervisor agent — orchestrator node

`supervisor_agent` **is** the LangGraph node. It reads `BlogState`, decides which specialist is still needed, and **calls** `research.invoke(...)` / `writer.invoke(...)` in process. That is “supervision by invocation.”

The routing here is **rule-based** (easy to read). You can later replace the `if` sequence with an LLM that picks the next specialist name; the sub-agent classes stay the same.

`TavilySearchResults` reads `TAVILY_API_KEY` from the environment.

In [11]:
class ResearchAgent:
    """Web research only — fills `research_notes` via Tavily."""

    name = "ResearchAgent"

    def invoke(self, state: BlogState) -> dict:
        topic = state["topic"]
        if not os.getenv("TAVILY_API_KEY"):
            raise RuntimeError("Missing TAVILY_API_KEY in environment (.env).")

        tool = TavilySearchResults(max_results=5)
        raw = tool.invoke({"query": topic})

        chunks: list[str] = []
        for i, item in enumerate(raw, start=1):
            if isinstance(item, dict):
                url = item.get("url", "")
                content = item.get("content", "") or item.get("snippet", "")
                title = item.get("title", "")
                header = title or url or f"Source {i}"
                chunks.append(f"### {header}\n{content}\n\n({url})")
            else:
                chunks.append(str(item))

        research_notes = "\n\n---\n\n".join(chunks) if chunks else "(No results returned.)"
        return {"research_notes": research_notes}


class WritingAgent:
    """Draft + save — uses `topic` and `research_notes`; writes `temp/blog-<slug>.md`."""

    name = "WritingAgent"
    SYSTEM = """You are an expert blog writer.
Write a clear, well-structured markdown article suitable for a technical audience.
Use headings (##, ###), short paragraphs, and a brief conclusion.
Do not invent specific statistics or quotes that are not supported by the research notes;
you may paraphrase and synthesize what is there.
"""

    def invoke(self, state: BlogState) -> dict:
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("Missing OPENAI_API_KEY in environment (.env).")

        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
        user = (
            f"Topic: {state['topic']}\n\n"
            "Research notes (from web search):\n"
            f"{state.get('research_notes', '')}\n\n"
            "Write the full blog post in markdown."
        )
        response = llm.invoke(
            [SystemMessage(content=self.SYSTEM), HumanMessage(content=user)]
        )
        blog_markdown = (
            response.content if isinstance(response.content, str) else str(response.content)
        )

        path = TEMP_BLOG_DIR / f"blog-{safe_slug(state['topic'])}.md"
        path.write_text(blog_markdown, encoding="utf-8")

        return {"blog_markdown": blog_markdown, "blog_file_path": str(path)}


def supervisor_agent(state: BlogState) -> dict:
    """Only LangGraph node: invokes sub-agents based on what state still needs."""
    research = ResearchAgent()
    writer = WritingAgent()
    trace: list[str] = list(state.get("supervisor_trace", []))
    merged: dict = dict(state)
    updates: dict = {}

    if not (merged.get("research_notes") or "").strip():
        trace.append(f"Supervisor invokes **{research.name}** (Tavily).")
        patch = research.invoke(merged)
        merged.update(patch)
        updates.update(patch)

    if not (merged.get("blog_markdown") or "").strip():
        trace.append(f"Supervisor invokes **{writer.name}** (LLM + file).")
        patch = writer.invoke(merged)
        merged.update(patch)
        updates.update(patch)

    trace.append("Supervisor: pipeline complete.")
    updates["supervisor_trace"] = trace
    return updates

### 4. Build the graph

The graph is intentionally **minimal**: one node (`supervisor_agent`). The **multi-agent** behavior lives **inside** that node via `.invoke()` on `ResearchAgent` and `WritingAgent`.

- `add_node("supervisor_agent", ...)` — the orchestrator.
- `START → supervisor_agent → END` — no separate nodes for research or writing.

`compile()` returns a runnable app you call with `.invoke(initial_state(...))`.

In [12]:
def build_blog_graph():
    graph = StateGraph(BlogState)
    graph.add_node("supervisor_agent", supervisor_agent)
    graph.add_edge(START, "supervisor_agent")
    graph.add_edge("supervisor_agent", END)
    return graph.compile()


blog_app = build_blog_graph()

### 5. Run the pipeline

Change `TOPIC` to any subject you want. After the run, open the printed path or browse `temp/` in the repo.

In [14]:
TOPIC = "Why multi-agent orchestration matters for product teams"

result = blog_app.invoke(initial_state(TOPIC))

print("--- Supervisor trace ---")
for line in result["supervisor_trace"]:
    print(line)
print()
print("Saved blog file:", result["blog_file_path"])
print()
print("--- Preview (first 1200 chars) ---")
print(result["blog_markdown"][:1200])

--- Supervisor trace ---
Supervisor invokes **ResearchAgent** (Tavily).
Supervisor invokes **WritingAgent** (LLM + file).
Supervisor: pipeline complete.

Saved blog file: C:\Users\mistr\Documents\PP\Agents-101\temp\blog-why-multi-agent-orchestration-matters-for-produc.md

--- Preview (first 1200 chars) ---
# Why Multi-Agent Orchestration Matters for Product Teams

In today's rapidly evolving business landscape, product teams are increasingly recognizing the importance of multi-agent orchestration. This approach not only streamlines operations but also enhances the ability to tackle complex problems that arise in various domains. In this article, we will explore what multi-agent orchestration is, its key benefits, and why it is essential for product teams aiming to leverage artificial intelligence (AI) effectively.

## What is Multi-Agent Orchestration?

Multi-agent orchestration refers to the coordinated management of multiple AI agents so they operate as a unified, goal-driven system.